# Notebook 11 — Relational-Hallucination Evaluation (paper §5–§6)

The core experiments for the paper:

1. **Incidence** — how often recommendation rationales make *relational* claims.
2. **The blind-spot finding (§5)** — text verifiers (MiniCheck/NLI) miss relational
   hallucinations that they catch for atomic claims.
3. **Head-to-head (§6, Table `tab:headtohead`)** — graph-grounded verification vs three
   baselines (MiniCheck as retrieval-grounded fact-checking, a dedicated NLI entailment
   model, and a citation-existence check) on the relational slice, scored against the
   **human gold**, with paired-bootstrap significance, coverage, and the edge-recall ceiling.
4. *(optional)* generator comparison (gpt-4o-mini v1.0 vs Llama-3.3-70B v1.1).

**Inputs (produce these first):**
- `agentic-nb2` → `scireqa_full_v1.1_humanmerged.jsonl` (carries `human_label`, `human_type`).
- `04_citation_graph_gnn` → `citecheck_results.json` (CiteCheck verdicts per claim; §5c
  recomputes these fresh from `citecheck_refs_cache.json` instead of trusting this file, since
  it can silently fall behind the benchmark -- see §5c for the bug this fixed).
The cells degrade gracefully and print what is missing.

In [3]:
# == 1. Load inputs (relational benchmark + CiteCheck verdicts) ===============
import json
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

def _find(name):
    for root in ["/kaggle/input/datasets/joytuntonny/human-eval", "/kaggle/working", "..", "."]:
        r = Path(root)
        if r.exists():
            hits = list(r.rglob(name))
            if hits:
                return hits[0]
    return None

# prefer human-merged relational gold; fall back to plain relational, then atomic
BENCH_PATH = (_find("scireqa_full_relational_v1.1_humanmerged.jsonl")
              or _find("scireqa_full_relational_v1.1.jsonl")
              or _find("scireqa_full_v1.1_humanmerged.jsonl")
              or _find("scireqa_full_v1.1.jsonl"))
assert BENCH_PATH, "benchmark not found - attach the dataset with scireqa_full_relational_v1.1*.jsonl"
HAS_HUMAN = "humanmerged" in BENCH_PATH.name
print("Benchmark:", BENCH_PATH, "| has_human:", HAS_HUMAN)
bench = [json.loads(l) for l in open(BENCH_PATH, encoding="utf-8") if l.strip()]
print("instances:", len(bench))

# CiteCheck verdicts keyed by "id|claim_id" (from 01_citecheck)
CITE_PATH = _find("citecheck_results.json")
citecheck = {}
if CITE_PATH:
    for r in json.load(open(CITE_PATH, encoding="utf-8")):
        citecheck[f"{r['id']}|{r['claim_id']}"] = r["verdict"]
print("CiteCheck verdicts:", len(citecheck), "| (0 -> run 01_citecheck first)")

Benchmark: /kaggle/input/datasets/joytuntonny/human-eval/scireqa_full_relational_v1.1_humanmerged.jsonl | has_human: True
instances: 10000
CiteCheck verdicts: 11517 | (0 -> run 01_citecheck first)


In [4]:
# == 2. Flatten to a per-claim table ==========================================
def is_text_halluc(v):       # a text verifier "detects hallucination"
    return v == "unsupported"

rows = []
for inst in bench:
    dom = inst["domain"]
    for lbl in inst.get("hallucination_labels", []):
        rel = lbl.get("relational")
        key = f"{inst['id']}|{lbl['claim_id']}"
        rows.append({
            "key": key, "domain": dom,
            "auto_relational": bool(rel),
            "relation": (rel or {}).get("relation"),
            "text_verdict": lbl.get("verdict"),          # MiniCheck / NLI verdict (silver)
            "text_backend": lbl.get("label_type"),
            "citecheck_verdict": citecheck.get(key),     # graph verdict (may be None)
            "human_label": lbl.get("human_label"),        # gold verdict (subset only)
            "human_type":  lbl.get("human_type"),         # gold type (subset only)
        })
print(f"claims: {len(rows)}")
print("auto-relational:", sum(r['auto_relational'] for r in rows),
      f"({100*sum(r['auto_relational'] for r in rows)/max(1,len(rows)):.1f}%)")
gold = [r for r in rows if r["human_type"]]   # human-verified subset
print("human-verified claims:", len(gold))

claims: 44310
auto-relational: 26245 (59.2%)
human-verified claims: 303


In [5]:
# == 3. Finding 1 — relational incidence ======================================
print("Relational-claim incidence")
overall = 100*sum(r['auto_relational'] for r in rows)/max(1,len(rows))
print(f"  overall: {overall:.1f}%")
for dom in sorted({r['domain'] for r in rows}):
    d = [r for r in rows if r['domain']==dom]
    print(f"  {dom:14s}: {100*sum(x['auto_relational'] for x in d)/max(1,len(d)):.1f}%  (n={len(d)})")
print("  relation types:", dict(Counter(r['relation'] for r in rows if r['auto_relational'])))

Relational-claim incidence
  overall: 59.2%
  acl_anthology : 58.6%  (n=8850)
  arxiv_cs      : 60.4%  (n=26777)
  pubmed        : 56.2%  (n=8683)
  relation types: {'builds_on': 13885, 'compares': 7573, 'shares_topic': 4520, 'supersedes': 267}


In [6]:
# == 4. Finding 2 — the text-verifier blind spot (needs human gold) ===========
# Gold positive = human marked the claim a hallucination.
#   relational hallucination : human_type == "relational" AND human_label == "unsupported"
#   atomic   hallucination   : human_label == "unsupported" and human_type != "relational"
if not gold:
    print("No human-verified labels yet -> run agentic-nb2 first.")
else:
    rel_pos = [r for r in gold if r["human_type"] == "relational" and r["human_label"] == "unsupported"]
    atom_pos = [r for r in gold if r["human_label"] == "unsupported" and r["human_type"] != "relational"]
    def recall(pos):
        return (100*sum(is_text_halluc(r["text_verdict"]) for r in pos)/len(pos)) if pos else float("nan")
    print(f"Text-verifier recall on ATOMIC hallucinations    : {recall(atom_pos):.1f}%  (n={len(atom_pos)})")
    print(f"Text-verifier recall on RELATIONAL hallucinations: {recall(rel_pos):.1f}%  (n={len(rel_pos)})")
    print("  -> the gap is the blind spot (expect relational << atomic).")

Text-verifier recall on ATOMIC hallucinations    : 0.0%  (n=19)
Text-verifier recall on RELATIONAL hallucinations: 0.0%  (n=62)
  -> the gap is the blind spot (expect relational << atomic).


In [7]:
# == 5. CiteCheck head-to-head on the relational gold slice ===================
def prf(pred_pos, gold_pos, universe):
    tp = len(pred_pos & gold_pos); fp = len(pred_pos - gold_pos); fn = len(gold_pos - pred_pos)
    P = tp/(tp+fp) if tp+fp else 0.0
    R = tp/(tp+fn) if tp+fn else 0.0
    F = 2*P*R/(P+R) if P+R else 0.0
    return P, R, F

# The relational gold slice must require BOTH: the auto-extractor found a checkable
# (paper_a, relation, paper_b) tuple, AND the human independently confirmed the claim is
# genuinely relational (human_type == "relational"). Using auto_relational alone (the
# original filter) let in claims the annotation guide explicitly tells annotators to
# REJECT as relational (Section 5: "both papers explore X" style topical statements that
# happen to trigger the cue-matcher's "extends"/"builds on" keywords) -- 28 of 178 such
# claims were in the old universe. Scoring CiteCheck against claims your own annotators
# say aren't real relational dependencies deflates precision for the wrong reason.
rel_gold = [r for r in gold if r["auto_relational"] and r["human_type"] == "relational"]
if not rel_gold:
    print("No relational gold claims yet -> run agentic-nb2 (with relational labels) first.")
else:
    universe   = {r["key"] for r in rel_gold}
    gold_pos   = {r["key"] for r in rel_gold if r["human_type"] == "relational" and r["human_label"] == "unsupported"}  # true relational hallucinations
    text_pos   = {r["key"] for r in rel_gold if is_text_halluc(r["text_verdict"])}
    cite_pos   = {r["key"] for r in rel_gold if r["citecheck_verdict"] == "unsupported"}
    print(f"Relational gold claims: {len(rel_gold)} | true relational hallucinations: {len(gold_pos)}")
    for name, pred in [("Text verifier (MiniCheck/NLI)", text_pos), ("CiteCheck (graph)", cite_pos)]:
        P, R, F = prf(pred, gold_pos, universe)
        print(f"  {name:32s}: P={P:.3f} R={R:.3f} F1={F:.3f}")
    cov = sum(1 for r in rel_gold if r["citecheck_verdict"] not in (None, "unverifiable")) / len(rel_gold)
    print(f"  CiteCheck coverage (adjudicable): {cov:.1%}")

Relational gold claims: 150 | true relational hallucinations: 32
  Text verifier (MiniCheck/NLI)   : P=0.000 R=0.000 F1=0.000
  CiteCheck (graph)               : P=0.276 R=0.500 F1=0.356
  CiteCheck coverage (adjudicable): 48.7%


## 5b. LLM-judge baseline (gpt-4o-mini) — a real text-only verifier

MiniCheck's raw support probability on these (real, non-synthetic) claims collapses into a
narrow band around 0.5 (0.46-0.53 -- see the diagnosis in `agentic-nb1b`, where the SAME
model gets AUC 0.87-0.98 on synthetic corruptions). That band is why the "Text verifier"
row above scores 0% recall on BOTH atomic and relational hallucinations -- an uninformative
raw signal, not proof text verifiers structurally can't do this at all.

This section adds a genuinely competent text-only judge: gpt-4o-mini, given the exact same
ground-truth-paper evidence MiniCheck used, asked directly whether each claim is supported.
Run on Kaggle: add `OPENAI_API_KEY` as a Kaggle Secret first (Add-ons -> Secrets).

In [8]:
# == 5b-i. LLM-judge setup (direct OpenAI API, gpt-4o-mini) ====================
import os, time, requests

try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    def _secret(k):
        try:
            return _sec.get_secret(k)
        except Exception:
            return ""
except Exception:
    def _secret(k):
        return os.getenv(k, "")

OPENAI_API_KEY = _secret("OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY", "")
assert OPENAI_API_KEY, "add OPENAI_API_KEY as a Kaggle Secret to run the LLM-judge baseline"

JUDGE_MODEL = "gpt-4o-mini"
JUDGE_OUT = Path("/kaggle/working/llm_judge_results.json") if Path("/kaggle").exists() else Path("../outputs/llm_judge_results.json")

def _build_evidence(gt_papers, max_chars=4000):
    """Identical construction to the pipeline's MiniCheck evidence (agentic-nb1 cell 23),
    so the LLM judge sees exactly what MiniCheck saw -- a fair, apples-to-apples baseline."""
    parts = []
    for p in (gt_papers or [])[:5]:
        title = (p.get("title", "") or "").strip()
        abst  = (p.get("abstract", "") or "").strip()
        year  = p.get("year", ""); venue = p.get("venue", "") or ""
        if abst:
            parts.append(f"{title} ({year}, {venue}): {abst[:400]}")
        elif title:
            parts.append(f"{title} ({year})")
    return " ".join(parts)[:max_chars]

_JUDGE_PROMPT = """You are fact-checking a single sentence from an AI-generated paper-recommendation rationale against the abstracts of the recommended papers.

Evidence (recommended papers' titles/abstracts):
{evidence}

Claim to check:
"{claim}"

Is this claim directly supported by the evidence above? If the claim describes a relationship between two papers (e.g., "X builds on Y", "X compares with Y"), only answer SUPPORTED if the evidence text itself states or clearly implies that specific relationship -- topical similarity alone is NOT enough.

Answer with exactly one word: SUPPORTED, UNSUPPORTED, or UNCERTAIN."""

def judge_claim(claim, evidence, retries=3):
    body = {
        "model": JUDGE_MODEL,
        "messages": [{"role": "user", "content": _JUDGE_PROMPT.format(evidence=evidence, claim=claim)}],
        "max_tokens": 5, "temperature": 0.0,
    }
    for attempt in range(retries):
        try:
            r = requests.post("https://api.openai.com/v1/chat/completions",
                              headers={"Authorization": f"Bearer {OPENAI_API_KEY}"}, json=body, timeout=30)
            r.raise_for_status()
            text = r.json()["choices"][0]["message"]["content"].strip().upper()
            if "UNSUPPORTED" in text: return "unsupported"
            if "SUPPORTED" in text:   return "supported"
            return "neutral"
        except Exception as e:
            if attempt == retries - 1:
                print(f"  [warn] judge call failed after {retries} tries: {e}")
                return "neutral"
            time.sleep(2 ** attempt)

print(f"LLM-judge baseline ready: {JUDGE_MODEL} via direct OpenAI API")

LLM-judge baseline ready: gpt-4o-mini via direct OpenAI API


In [9]:
# == 5b-ii. Run the judge over every human-verified claim (303 claims, checkpointed) ==
judge_cache = {}
if JUDGE_OUT.exists():
    judge_cache = json.loads(JUDGE_OUT.read_text(encoding="utf-8"))
    print(f"Resuming: {len(judge_cache)} claims already judged.")

gold_keys = {r["key"] for r in gold}
_ev_cache = {}
n_new = 0
for inst in bench:
    for lbl in inst.get("hallucination_labels", []):
        key = f"{inst['id']}|{lbl['claim_id']}"
        if key not in gold_keys or key in judge_cache:
            continue
        ev = _ev_cache.setdefault(inst["id"], _build_evidence(inst.get("ground_truth_papers", [])))
        judge_cache[key] = judge_claim(lbl["claim_text"], ev)
        n_new += 1
        if n_new % 25 == 0:
            JUDGE_OUT.parent.mkdir(parents=True, exist_ok=True)
            JUDGE_OUT.write_text(json.dumps(judge_cache), encoding="utf-8")
            print(f"  judged {n_new} new claims so far...")

JUDGE_OUT.parent.mkdir(parents=True, exist_ok=True)
JUDGE_OUT.write_text(json.dumps(judge_cache), encoding="utf-8")
print(f"Done. {len(judge_cache)} total claims judged (of {len(gold_keys)} human-verified). Saved: {JUDGE_OUT}")

Resuming: 303 claims already judged.
Done. 303 total claims judged (of 303 human-verified). Saved: /kaggle/working/llm_judge_results.json


In [10]:
# == 5b-iii. Re-run Finding 2 (blind spot) + Finding 3 (head-to-head) with the LLM judge ==
def llm_verdict(key):
    return judge_cache.get(key)

print("=== Finding 2 (blind spot): MiniCheck vs LLM judge ===")
for name, vfn in [("Text verifier (MiniCheck)", lambda r: r["text_verdict"]),
                   ("LLM judge (gpt-4o-mini)",  lambda r: llm_verdict(r["key"]))]:
    def recall(pos):
        scored = [r for r in pos if vfn(r) is not None]
        return (100*sum(is_text_halluc(vfn(r)) for r in scored)/len(scored)) if scored else float("nan")
    print(f"  {name}")
    print(f"    ATOMIC     recall: {recall(atom_pos):.1f}%  (n={len(atom_pos)})")
    print(f"    RELATIONAL recall: {recall(rel_pos):.1f}%  (n={len(rel_pos)})")

print("\n=== Finding 3 (head-to-head) on the relational gold slice: MiniCheck vs LLM judge vs CiteCheck ===")
universe = {r["key"] for r in rel_gold}
gold_pos = {r["key"] for r in rel_gold if r["human_type"] == "relational" and r["human_label"] == "unsupported"}
text_pos = {r["key"] for r in rel_gold if is_text_halluc(r["text_verdict"])}
llm_pos  = {r["key"] for r in rel_gold if is_text_halluc(llm_verdict(r["key"]))}
cite_pos = {r["key"] for r in rel_gold if r["citecheck_verdict"] == "unsupported"}
print(f"Relational gold claims: {len(rel_gold)} | true relational hallucinations: {len(gold_pos)}")
for name, pred in [("Text verifier (MiniCheck)", text_pos),
                   ("LLM judge (gpt-4o-mini)",  llm_pos),
                   ("CiteCheck (graph)",        cite_pos)]:
    P, R, F = prf(pred, gold_pos, universe)
    print(f"  {name:28s}: P={P:.3f} R={R:.3f} F1={F:.3f}")

=== Finding 2 (blind spot): MiniCheck vs LLM judge ===
  Text verifier (MiniCheck)
    ATOMIC     recall: 0.0%  (n=19)
    RELATIONAL recall: 0.0%  (n=62)
  LLM judge (gpt-4o-mini)
    ATOMIC     recall: 21.1%  (n=19)
    RELATIONAL recall: 25.8%  (n=62)

=== Finding 3 (head-to-head) on the relational gold slice: MiniCheck vs LLM judge vs CiteCheck ===
Relational gold claims: 150 | true relational hallucinations: 32
  Text verifier (MiniCheck)   : P=0.000 R=0.000 F1=0.000
  LLM judge (gpt-4o-mini)     : P=0.312 R=0.312 F1=0.312
  CiteCheck (graph)           : P=0.276 R=0.500 F1=0.356


## 5c. Refreshing CiteCheck + the citation-existence-check baseline (for Table `tab:headtohead`)

**Bug found while building this section:** `citecheck_results.json` (from `01_citecheck`) was
missing verdicts for 77 of the 149 claims in `rel_gold` -- not "unverifiable", just absent,
because `01_citecheck` was last run against an earlier snapshot of the benchmark with fewer
resolvable `(paper_a, paper_b)` tuples (human review since then filled in / corrected some
endpoints). Relying on the stale file silently understated CiteCheck's coverage (63.8% -> the
true current figure is higher) and, for 2 claims, scored a verdict against endpoints a human
annotator had since corrected. Rather than depend on a snapshot that can silently drift out of
sync with the benchmark again, this cell recomputes CiteCheck verdicts **fresh**, every run,
directly from `rel_gold`'s own `(paper_a, relation, paper_b)` tuples -- fetching any endpoint
missing from the cache via the same Semantic Scholar batch call `01_citecheck` uses, so this
section is self-healing regardless of what `citecheck_results.json` currently contains.

This also gives us, for free, the raw resolution status per endpoint -- exactly what a
**citation-existence check** (entity-level verification, `\citep{khajavi2026citecheck}`-style:
does the reference exist and resolve to a real paper?) would use. That baseline flags a claim
only when at least one endpoint fails to resolve; it has no mechanism to assess whether the
*relationship* between two real, existing papers is accurate, so it is expected to recover only
the small slice of relational hallucinations that happen to also involve an unresolved
endpoint.

In [11]:
# == 5c-i. Fresh CiteCheck recomputation + citation-existence-check baseline ===
import re, time, requests as _requests

REFS_CACHE_PATH = _find("citecheck_refs_cache.json") or (
    Path("/kaggle/working/citecheck_refs_cache.json") if Path("/kaggle").exists()
    else Path("../data/citecheck_refs_cache.json"))
refs_cache = json.loads(REFS_CACHE_PATH.read_text(encoding="utf-8")) if REFS_CACHE_PATH.exists() else {}
print(f"refs cache: {len(refs_cache)} papers ({REFS_CACHE_PATH})")

ARXIV_RE = re.compile(r"^\d{4}\.\d{4,5}")
def _s2_id(pid):
    pid = str(pid)
    return "ARXIV:" + pid if ARXIV_RE.match(pid) else pid

# rel_gold rows don't carry paper_a/paper_b directly (see cell 2) -- rebuild from bench.
tuple_of = {}
for inst in bench:
    for lbl in inst.get("hallucination_labels", []):
        rel = lbl.get("relational")
        key = f"{inst['id']}|{lbl['claim_id']}"
        if rel and rel.get("paper_a") and rel.get("paper_b"):
            tuple_of[key] = (rel["paper_a"], rel["relation"], rel["paper_b"])

all_papers = sorted({p for k in universe if k in tuple_of for p in (tuple_of[k][0], tuple_of[k][2])})
todo = [p for p in all_papers if p not in refs_cache]
print(f"{len(all_papers)} unique endpoint papers in rel_gold | {len(todo)} need fresh S2 lookup")

if todo:
    S2_BASE = "https://api.semanticscholar.org/graph/v1"
    got = {}
    for i in range(0, len(todo), 100):
        sub = todo[i:i+100]
        for attempt in range(5):
            try:
                rr = _requests.post(f"{S2_BASE}/paper/batch", params={"fields": "paperId,year,references.paperId"},
                                     json={"ids": [_s2_id(p) for p in sub]}, timeout=60)
                if rr.status_code == 200:
                    for pid, obj in zip(sub, rr.json()):
                        got[pid] = obj
                    break
                elif rr.status_code == 429:
                    time.sleep(5*(attempt+1))
                else:
                    time.sleep(2)
            except Exception:
                time.sleep(2)
        time.sleep(0.4)
    for raw, obj in got.items():
        obj = obj or {}
        refs_cache[raw] = {"pid": obj.get("paperId"), "year": obj.get("year"),
                            "refs": [x.get("paperId") for x in (obj.get("references") or []) if x.get("paperId")]}
    REFS_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    REFS_CACHE_PATH.write_text(json.dumps(refs_cache), encoding="utf-8")
    print(f"resolved {sum(1 for p in todo if refs_cache.get(p, {}).get('pid'))}/{len(todo)} new papers -> cache saved")

# in-memory graph structures (identical construction to 01_citecheck cell 3)
refs, year, pid_of = {}, {}, {}
for raw in all_papers:
    e = refs_cache.get(raw) or {}
    if e.get("pid"):
        pid_of[raw] = e["pid"]; year[raw] = e.get("year")
        refs[raw] = set(e.get("refs") or []) | {e["pid"]}
print(f"resolved {len(pid_of)}/{len(all_papers)} endpoint papers")

def cites(a, b):
    return bool(pid_of.get(b) and a in refs and pid_of[b] in refs[a])
def shared_refs(a, b):
    return len(refs.get(a, set()) & refs.get(b, set())) if (a in refs and b in refs) else 0

COCITE_MIN = 1  # calibrated on human gold via 5-fold CV (see 01_citecheck cell 4)
def verify_relation(a, relation, b):
    if a not in refs or b not in refs:
        return "unverifiable"
    a_b, b_a = cites(a, b), cites(b, a)
    sh = shared_refs(a, b)
    ya, yb = year.get(a), year.get(b)
    if relation in ("builds_on", "uses_method_of", "supersedes"):
        if a_b and (ya is None or yb is None or ya >= yb):
            return "supported"
        if b_a:
            return "supported"
        return "unsupported"
    if relation in ("compares", "shares_topic"):
        return "supported" if (a_b or b_a or sh >= COCITE_MIN) else "unsupported"
    return "unverifiable"

def e_indicator(a, b):
    if a not in refs or b not in refs:
        return None
    return int(cites(a, b) or cites(b, a) or shared_refs(a, b) >= COCITE_MIN)

cite_verdict_fresh, e_ab = {}, {}
for k in universe:
    if k in tuple_of:
        a, rel, b = tuple_of[k]
        cite_verdict_fresh[k] = verify_relation(a, rel, b)
        e_ab[k] = e_indicator(a, b)
    else:
        cite_verdict_fresh[k] = "unverifiable"   # no (paper_a, paper_b) tuple extracted at all
        e_ab[k] = None

cite_pos = {k for k in universe if cite_verdict_fresh[k] == "unsupported"}
adjudicable = {k for k in universe if cite_verdict_fresh[k] not in ("unverifiable",)}
coverage = len(adjudicable) / len(universe)
print(f"\nCiteCheck (fresh) verdict distribution:", Counter(cite_verdict_fresh.values()))
print(f"Coverage: {coverage:.1%} ({len(adjudicable)}/{len(universe)})")

# citation-existence-check baseline: flags a claim iff an endpoint fails to resolve/wasn't
# even extracted -- it cannot see the relationship itself.
exists_pos = {k for k in universe if cite_verdict_fresh[k] == "unverifiable"}
P, R, F = prf(exists_pos, gold_pos, universe)
print(f"\nCitation-existence check: P={P:.3f} R={R:.3f} F1={F:.3f}  (flags {len(exists_pos)}/{len(universe)} claims)")

P, R, F = prf(cite_pos, gold_pos, universe)
print(f"CiteCheck (fresh, ours)  : P={P:.3f} R={R:.3f} F1={F:.3f}")

# EdgeRecallCeiling (Eq. eq:edgerecallceiling): over claims experts confirm are genuinely
# TRUE (human_label == "supported"), what fraction still have an edge at all?
T = [r for r in rel_gold if r["human_label"] == "supported"]
T_with_e = [r for r in T if e_ab.get(r["key"]) == 1]
edge_recall_ceiling = len(T_with_e) / len(T) if T else float("nan")
print(f"\nEdgeRecallCeiling: {len(T_with_e)}/{len(T)} = {edge_recall_ceiling:.1%}")

refs cache: 21191 papers (/kaggle/input/datasets/joytuntonny/human-eval/citecheck_refs_cache.json)
139 unique endpoint papers in rel_gold | 0 need fresh S2 lookup
resolved 139/139 endpoint papers

CiteCheck (fresh) verdict distribution: Counter({'unverifiable': 77, 'unsupported': 54, 'supported': 19})
Coverage: 48.7% (73/150)

Citation-existence check: P=0.195 R=0.469 F1=0.275  (flags 77/150 claims)
CiteCheck (fresh, ours)  : P=0.296 R=0.500 F1=0.372

EdgeRecallCeiling: 30/113 = 26.5%


## 5d. Natural-language-inference baseline (DeBERTa-v3 MNLI+FEVER+ANLI)

The MiniCheck/LLM-judge baselines above both use an LLM (fine-tuned or prompted) to *check a
claim against retrieved evidence*, which the paper's Related Work groups under
"retrieval-grounded fact-checking." Table `tab:headtohead` also needs a baseline from the
older, distinct **natural-language-inference** family: a dedicated entailment classifier
(not an LLM) that scores whether a premise entails a hypothesis. We use
`MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli` \citep{laurer2024less}, a standard, widely used
off-the-shelf NLI checkpoint, with premise = the same evidence MiniCheck/the LLM judge saw and
hypothesis = the claim text. Entailment probability above 0.5 -> `supported`; contradiction
probability above 0.5 -> `unsupported`; otherwise `neutral` -- the same three-way verdict space
`is_text_halluc()` already expects, so it plugs directly into Finding 2/3's existing scoring.

No API key needed (local/Kaggle GPU or CPU inference only); 303 claims run in well under a
minute on a GPU and a few minutes on CPU.

In [12]:
# == 5d-i. NLI setup + run over every human-verified claim (checkpointed) =======
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

NLI_MODEL_NAME = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
NLI_OUT = Path("/kaggle/working/nli_results.json") if Path("/kaggle").exists() else Path("../outputs/nli_results.json")

_nli_device = "cuda" if torch.cuda.is_available() else "cpu"
_nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
_nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME).to(_nli_device).eval()
_id2label = {i: l.lower() for i, l in _nli_model.config.id2label.items()}
print("NLI model loaded:", NLI_MODEL_NAME, "on", _nli_device, "| labels:", _id2label)

def nli_verdict(premise, hypothesis):
    if not premise.strip() or not hypothesis.strip():
        return "neutral"
    with torch.no_grad():
        inputs = _nli_tok(premise, hypothesis, truncation=True, max_length=512, return_tensors="pt").to(_nli_device)
        probs = torch.softmax(_nli_model(**inputs).logits, dim=-1)[0].tolist()
    scores = {_id2label[i]: p for i, p in enumerate(probs)}
    if scores.get("entailment", 0) > 0.5:
        return "supported"
    if scores.get("contradiction", 0) > 0.5:
        return "unsupported"
    return "neutral"

nli_cache = {}
if NLI_OUT.exists():
    nli_cache = json.loads(NLI_OUT.read_text(encoding="utf-8"))
    print(f"Resuming: {len(nli_cache)} claims already scored.")

gold_keys = {r["key"] for r in gold}
_ev_cache = {}
n_new = 0
for inst in bench:
    for lbl in inst.get("hallucination_labels", []):
        key = f"{inst['id']}|{lbl['claim_id']}"
        if key not in gold_keys or key in nli_cache:
            continue
        ev = _ev_cache.setdefault(inst["id"], _build_evidence(inst.get("ground_truth_papers", [])))
        nli_cache[key] = nli_verdict(ev, lbl["claim_text"])
        n_new += 1
        if n_new % 50 == 0:
            NLI_OUT.parent.mkdir(parents=True, exist_ok=True)
            NLI_OUT.write_text(json.dumps(nli_cache), encoding="utf-8")
            print(f"  scored {n_new} new claims so far...")

NLI_OUT.parent.mkdir(parents=True, exist_ok=True)
NLI_OUT.write_text(json.dumps(nli_cache), encoding="utf-8")
print(f"Done. {len(nli_cache)} total claims scored (of {len(gold_keys)} human-verified). Saved: {NLI_OUT}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NLI model loaded: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli on cuda | labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
  scored 50 new claims so far...
  scored 100 new claims so far...
  scored 150 new claims so far...
  scored 200 new claims so far...
  scored 250 new claims so far...
  scored 300 new claims so far...
Done. 303 total claims scored (of 303 human-verified). Saved: /kaggle/working/nli_results.json


In [13]:
# == 5e. Full head-to-head for Table `tab:headtohead` (+ paired bootstrap significance) ==
nli_pos = {k for k in universe if is_text_halluc(nli_cache.get(k))}

systems = {
    "Natural language inference":       nli_pos,
    "Retrieval-grounded fact-checking": text_pos,
    "Citation-existence check":         exists_pos,
    "\\method{} (ours)":                cite_pos,
}
print(f"Relational gold claims: {len(universe)} | true relational hallucinations: {len(gold_pos)}\n")
for name, pred in systems.items():
    P, R, F = prf(pred, gold_pos, universe)
    print(f"  {name:34s}: P={P:.3f} R={R:.3f} F1={F:.3f}")

print(f"\nCoverage: {coverage:.1%}  |  EdgeRecallCeiling: {edge_recall_ceiling:.1%}")

# Paired bootstrap (claim-level resampling, B=5000, seed=13) on F1, ours vs each baseline.
keys = sorted(universe)
gold_arr = np.array([1 if k in gold_pos else 0 for k in keys])
def _pred_arr(pred_set):
    return np.array([1 if k in pred_set else 0 for k in keys])

def _f1(pred, g):
    tp = np.sum((pred == 1) & (g == 1)); fp = np.sum((pred == 1) & (g == 0)); fn = np.sum((pred == 0) & (g == 1))
    P = tp/(tp+fp) if tp+fp > 0 else 0.0
    R = tp/(tp+fn) if tp+fn > 0 else 0.0
    return 2*P*R/(P+R) if (P+R) > 0 else 0.0

ours_arr = _pred_arr(cite_pos)
rng = np.random.default_rng(13)
B, n = 5000, len(keys)
print(f"\nPaired bootstrap (B={B}, seed=13), \\method{{}} vs each baseline:")
for name, pred in systems.items():
    if name == "\\method{} (ours)":
        continue
    pred_arr = _pred_arr(pred)
    diffs = np.empty(B)
    for i in range(B):
        idx = rng.integers(0, n, n)
        g = gold_arr[idx]
        diffs[i] = _f1(ours_arr[idx], g) - _f1(pred_arr[idx], g)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    p = min(2*min((diffs <= 0).mean(), (diffs >= 0).mean()), 1.0)
    print(f"  ours - {name:34s}: {diffs.mean():+.3f}  95% CI [{lo:+.3f},{hi:+.3f}]  p={p:.4f}")

Relational gold claims: 150 | true relational hallucinations: 32

  Natural language inference        : P=0.000 R=0.000 F1=0.000
  Retrieval-grounded fact-checking  : P=0.000 R=0.000 F1=0.000
  Citation-existence check          : P=0.195 R=0.469 F1=0.275
  \method{} (ours)                  : P=0.296 R=0.500 F1=0.372

Coverage: 48.7%  |  EdgeRecallCeiling: 26.5%

Paired bootstrap (B=5000, seed=13), \method{} vs each baseline:
  ours - Natural language inference        : +0.369  95% CI [+0.232,+0.490]  p=0.0000
  ours - Retrieval-grounded fact-checking  : +0.369  95% CI [+0.239,+0.495]  p=0.0000
  ours - Citation-existence check          : +0.095  95% CI [-0.114,+0.292]  p=0.3656


## 5f. Worked qualitative example — the blind spot (benchmark + CiteCheck), free

A single relational claim a **text verifier passed** but **CiteCheck flagged** (`unsupported`),
shown with both verdicts and the human gold. This is the evidence figure for the finding.
(The live, end-to-end system demo will live in the web app, not here.)

In [14]:
# Find a relational claim a TEXT verifier did NOT flag but CiteCheck flagged unsupported.
# Prefer a HUMAN-CONFIRMED true relational hallucination (human_type=="relational" and
# human_label=="unsupported") so the worked example is a proven catch, not just an
# illustration of the mechanism; among those, prefer text_verdict=="supported" for the
# starkest contrast (a confident, wrong verdict) over "neutral"/"general_statement".
_verdict_priority = {"supported": 0, "neutral": 1, "general_statement": 2}

def _find_example(require_human_gold):
    best = None
    for _inst in bench:
        for _lbl in _inst.get("hallucination_labels", []):
            _rel = _lbl.get("relational")
            if not (_rel and _rel.get("paper_a") and _rel.get("paper_b")):
                continue
            if require_human_gold and not (_lbl.get("human_type") == "relational"
                                            and _lbl.get("human_label") == "unsupported"):
                continue
            _cv = citecheck.get(f"{_inst['id']}|{_lbl['claim_id']}")
            _tv = _lbl.get("verdict")
            if _cv != "unsupported" or _tv not in _verdict_priority:
                continue
            _rank = _verdict_priority[_tv]
            if best is None or _rank < best[0]:
                best = (_rank, _inst, _lbl, _cv)
                if _rank == 0:
                    return best
    return best

_best = _find_example(require_human_gold=True)
_confirmed = _best is not None
if _best is None:
    _best = _find_example(require_human_gold=False)

if _best:
    _, _inst, _lbl, _cv = _best; _rel = _lbl["relational"]
    print("Instance:", _inst["id"], "| domain:", _inst.get("domain"),
          "| human-confirmed:", _confirmed)
    print("\nReference rationale (excerpt):\n", (_inst.get("reference_rationale", "") or "")[:600])
    print("\nRelational claim under test:")
    print("  claim    :", _lbl["claim_text"])
    print(f"  relation : ({_rel['paper_a']}) {_rel['relation']} ({_rel['paper_b']})")
    print("  text verifier (MiniCheck/NLI):", _lbl.get("verdict"))
    print("  CiteCheck (citation graph)   :", _cv, "  <- catches what the text verifier missed")
    print("  human gold                   :", _lbl.get("human_label") or "[pending annotation]",
          "/", _lbl.get("human_type") or "")
else:
    print("No blind-spot example yet - need citecheck_results.json + a relational claim (run 01_citecheck).")


Instance: f5de313e777b | domain: acl_anthology | human-confirmed: True

Reference rationale (excerpt):
 The recommended papers relate to each other and the query paper through their application of Natural Language Processing (NLP) techniques to various domains. Paper [2] "Transformer-Based Natural Language Processing Models for Mining Unstructured Oncology Clinical Notes to Improve Drug Matching" builds on the concept of using transformer models for NLP tasks, similar to paper [1] "Application of the transformer model algorithm in chinese word sense disambiguation: a case study in chinese language", which explores the application of transformer models to Chinese word sense disambiguation. Paper 

Relational claim under test:
  claim    : Paper [2] "Transformer-Based Natural Language Processing Models for Mining Unstructured Oncology Clinical Notes to Improve Drug Matching" builds on the concept of using transformer models for NLP tasks, similar to paper [1] "Application of the transfor

In [15]:
# == 6. (optional) Generator comparison: gpt-4o-mini (v1.0) vs Llama (v1.1) ====
# Compares relational incidence / hallucination rate across generators on the same task.
def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()] if Path(p).exists() else []

v10 = _find("scireqa_arxiv.jsonl")          # old gpt-4o-mini set (if kept)
v11 = _find("scireqa_arxiv_v1.1.jsonl")     # Llama-3.3-70B set
for tag, p in [("gpt-4o-mini (v1.0)", v10), ("llama-3.3-70b (v1.1)", v11)]:
    data = load_jsonl(p) if p else []
    if not data:
        print(f"{tag}: not found ({p})"); continue
    cl = [l for d in data for l in d.get("hallucination_labels", [])]
    rel = sum(1 for l in cl if l.get("relational"))
    uns = sum(1 for l in cl if l.get("verdict") == "unsupported")
    print(f"{tag}: claims={len(cl)} relational={100*rel/max(1,len(cl)):.1f}% "
          f"unsupported(text)={100*uns/max(1,len(cl)):.1f}%")

gpt-4o-mini (v1.0): not found (None)
llama-3.3-70b (v1.1): not found (None)


## 7. Summary

This notebook produces the paper's core results once `agentic-nb2` (human gold incl.
`relational` type) has been run:

- **§3** relational incidence (motivation),
- **§4** the text-verifier blind spot (finding),
- **§5** MiniCheck vs \method{} on the relational gold slice (method result),
- **§5b** LLM-judge (gpt-4o-mini) robustness check,
- **§5c** fresh CiteCheck recomputation (self-healing, fixes the coverage-staleness bug) +
  citation-existence-check baseline,
- **§5d** natural-language-inference baseline (DeBERTa-v3 MNLI),
- **§5e** the full head-to-head for Table `tab:headtohead`, with paired-bootstrap
  significance, coverage, and the edge-recall ceiling,
- **§5f** worked qualitative example,
- **§6** generator comparison (supporting finding).